# Tratamento de Dados para Fine-Tuning

## 1. Limpeza inicial (data_clean.py)
Carrega JSONL, remove linhas vazias/malformadas, conteúdos vazios, duplicados por conteúdo e textos muito curtos (<10 chars).

In [ ]:
# Replicação do conteúdo de data_clean.py (sem alterações de lógica)
import json
import os

def clean_data(input_file, output_file):
    data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Skipping malformed JSON on line {line_num}: {e}")
    original_count = len(data)
    print(f"Original data count: {original_count}")
    cleaned_data = [item for item in data if item.get('content') and item['content'].strip()]
    seen = set()
    deduped_data = []
    for item in cleaned_data:
        content = item['content'].strip()
        if content not in seen:
            seen.add(content)
            deduped_data.append(item)
    final_data = [item for item in deduped_data if len(item['content'].strip()) >= 10]
    final_count = len(final_data)
    print(f"Filtered data count (after removing empty, duplicates, and short reviews): {final_count}")
    print(f"Removed {original_count - final_count} entries")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    print(f"Cleaned data saved to {output_file}")

input_file = 'D:\\FIAP_TC3\\LF-Amazon-1.3M\\trn.json'
output_file = 'train_cleaned.json'

if os.path.exists(input_file):
    try:
        sample_n = 20000
        raw_sample = pd.read_json(input_file, lines=True, nrows=sample_n)
        print(f'Amostra carregada: {len(raw_sample)} linhas')
        print('Colunas:', list(raw_sample.columns))
        display(raw_sample.head(5))
        if 'content' in raw_sample.columns:
            raw_sample['content_len'] = raw_sample['content'].astype(str).str.len()
            print('\nDistribuição de tamanho de content (amostra):')
            print(raw_sample['content_len'].describe(percentiles=[.1,.25,.5,.75,.9,.95]))
            short_mask = raw_sample['content_len'] < 10
            print(f'Conteúdos muito curtos (<10 chars): {short_mask.sum()} ({short_mask.mean():.2%})')
            dup_content = raw_sample['content'].duplicated().sum()
            print(f'Duplicados por conteúdo na amostra: {dup_content} ({dup_content/len(raw_sample):.2%})')
            missing_content = raw_sample['content'].isna().sum()
            print(f'Content ausente (NaN): {missing_content}')
        if 'title' in raw_sample.columns:
            title_missing = raw_sample['title'].isna().sum()
            print(f'Títulos ausentes (NaN): {title_missing}')
    except Exception as e:
        print('Falha ao inspecionar com pandas:', e)

    clean_data(input_file, output_file)
else:
    print(f"Error: {input_file} not found. Skipping.")

Amostra carregada: 20000 linhas
Colunas: ['uid', 'title', 'content', 'target_ind', 'target_rel']


,uid,title,content,target_ind,target_rel
0,0000031909,Girls Ballet Tutu Neon Pink,High quality 3 layer ballet tutu. 12 inches in...,"[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 2...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,0000032034,Adult Ballet Tutu Yellow,,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 33, 36, 37,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,0000913154,The Way Things Work: An Illustrated Encycloped...,,"[116, 117, 118, 119, 120, 121, 122]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
3,0001360000,Mog's Kittens,Judith Kerr&#8217;s best&#8211;selling adventu...,"[146, 147, 148, 149, 495]","[1.0, 1.0, 1.0, 1.0, 1.0]"
4,0001381245,Misty of Chincoteague,,[151],[1.0]



Distribuição de tamanho de content (amostra):
count    20000.000000
mean       789.261550
std        885.309039
min          0.000000
10%          0.000000
25%        152.000000
50%        605.500000
75%       1238.000000
90%       1637.000000
95%       1968.000000
max      33378.000000
Name: content_len, dtype: float64
Conteúdos muito curtos (<10 chars): 3678 (18.39%)
Duplicados por conteúdo na amostra: 5037 (25.19%)
Content ausente (NaN): 0
Títulos ausentes (NaN): 0
Original data count: 2248619
Original data count: 2248619
Filtered data count (after removing empty, duplicates, and short reviews): 1271203
Removed 977416 entries
Filtered data count (after removing empty, duplicates, and short reviews): 1271203
Removed 977416 entries
Cleaned data saved to train_cleaned.json
Cleaned data saved to train_cleaned.json


In [5]:
# Amostra após limpeza
import pandas as pd, json
if os.path.exists('train_cleaned.json'):
    with open('train_cleaned.json','r',encoding='utf-8') as f:
        df_clean = pd.DataFrame(json.load(f))
    display(df_clean.head(5))
else:
    print('Arquivo train_cleaned.json não encontrado.')

,uid,title,content,target_ind,target_rel
0,0000031909,Girls Ballet Tutu Neon Pink,High quality 3 layer ballet tutu. 12 inches in...,"[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 2...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,0001360000,Mog's Kittens,Judith Kerr&#8217;s best&#8211;selling adventu...,"[146, 147, 148, 149, 495]","[1.0, 1.0, 1.0, 1.0, 1.0]"
2,0000031895,Girls Ballet Tutu Neon Blue,Dance tutu for girls ages 2-8 years. Perfect f...,"[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 2...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
3,000100039X,The Prophet,"In a distant, timeless place, a mysterious pro...","[329, 330, 331, 332, 333, 334, 335, 336, 337, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
4,0001473905,Rightly Dividing the Word,--This text refers to thePaperbackedition.,"[181, 182, 307, 380, 381]","[1.0, 1.0, 1.0, 1.0, 1.0]"


## 2. Exploração e agregação de títulos
Agrupa itens por título, consolida conteúdos duplicados e salva train_cleaned_unique.json.

In [8]:
# Replicação de exploration.py
import json
from collections import defaultdict

try:
    with open('train_cleaned.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print('Error: train_cleaned.json not found.')
    data = []
except json.JSONDecodeError:
    print('Error: Invalid JSON format.')
    data = []

title_groups = defaultdict(list)
for item in data:
    title = item.get('title', '')
    content = item.get('content', '')
    title_groups[title].append(content)

cleaned_data = []
duplicates_count = 0
for title, contents in title_groups.items():
    if len(contents) > 1:
        duplicates_count += len(contents) - 1
        merged_content = '\n'.join(contents)
    else:
        merged_content = contents[0] if contents else ''
    cleaned_data.append({'title': title, 'content': merged_content})

print(f"Original entries: {len(data)}")
print(f"Duplicates merged: {duplicates_count}")
print(f"Unique titles after cleaning: {len(cleaned_data)}")

with open('train_cleaned_unique.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=4)
print('Cleaned data saved to train_cleaned_unique.json')


duplicated_titles = [(t, c) for t, c in title_groups.items() if len(c) > 1]

if duplicated_titles:
    sample_dups = duplicated_titles[:5]
    rows = []
    for t, contents in sample_dups:
        merged = '\n'.join(contents)
        rows.append({
            'title': t,
            'qtd_itens_agrupados': len(contents),
            'tamanho_conteudo_unido': len(merged),
            'conteudo_unido_preview': merged[:300].replace('\n', ' \\n ')
        })
    display(pd.DataFrame(rows))
else:
    print('Nenhum título duplicado encontrado para amostra.')

Original entries: 1271203
Duplicates merged: 114793
Unique titles after cleaning: 1156410
Cleaned data saved to train_cleaned_unique.json


,title,qtd_itens_agrupados,tamanho_conteudo_unido,conteudo_unido_preview
0,The Prophet,3,14001,"In a distant, timeless place, a mysterious pro..."
1,The Book of Revelation,2,3344,"American Baptist pastor, Bible teacher, and wr..."
2,Harlequin,2,864,"Morris West, an Australian by birth, has lived..."
3,Circus,3,1277,Great reading' Sunday Telegraph 'An action-pac...
4,Passenger to Frankfurt,2,763,'Marvellously entertaining' OBSERVER 'It is no...


In [9]:
# Amostra após agregação por título
import pandas as pd, json, os
if os.path.exists('train_cleaned_unique.json'):
    with open('train_cleaned_unique.json','r',encoding='utf-8') as f:
        df_unique = pd.DataFrame(json.load(f))
    display(df_unique.head(5))
else:
    print('Arquivo train_cleaned_unique.json não encontrado.')

,title,content
0,Girls Ballet Tutu Neon Pink,High quality 3 layer ballet tutu. 12 inches in...
1,Mog's Kittens,Judith Kerr&#8217;s best&#8211;selling adventu...
2,Girls Ballet Tutu Neon Blue,Dance tutu for girls ages 2-8 years. Perfect f...
3,The Prophet,"In a distant, timeless place, a mysterious pro..."
4,Rightly Dividing the Word,--This text refers to thePaperbackedition.


## 3. Preparação dataset

In [13]:
# Replicação (já sem filtro de idioma).
import sys, json, re, unicodedata, argparse, random, hashlib, statistics, os
from pathlib import Path
from typing import List, Dict, Any, Tuple
from collections import Counter
sys.argv = ['prepare_product_lora_dataset.py','--seed','42','--min-tokens','15','--max-tokens','300']
try:
    import tiktoken
    TOKENIZER = tiktoken.get_encoding('cl100k_base')
except Exception:
    TOKENIZER = None
DISCLAIMER_REGEX = re.compile(r'(imagem meramente ilustrativa|oferta válida.*?estoques?|sujeito a alteração|todos os direitos reservados|imagem ilustrativa|produto sujeito a.*?disponibilidade|consulte.*?disponibilidade)', re.IGNORECASE)
INSTRUCTION_TEMPLATES_EN = [
  'Create a clear and attractive product description.',
]
def load_data_en(path: Path) -> List[Dict[str, Any]]:
    try:
        with open(path,'r',encoding='utf-8') as f: data = json.load(f)
        if not isinstance(data, list): print('ERROR: File must contain a list of JSON objects'); exit(1)
        print(f'INFO: Loaded {len(data)} records from file'); return data
    except FileNotFoundError: print(f'ERROR: File not found: {path}'); exit(1)
    except json.JSONDecodeError as e: print(f'ERROR: Invalid JSON: {e}'); exit(1)
    except Exception as e: print(f'ERROR: Failed to load file: {e}'); exit(1)
def normalize_text_en(txt: str) -> str:
    if not txt: return ''
    txt = unicodedata.normalize('NFKC', txt)
    txt = re.sub(r'<[^>]+>', ' ', txt)
    txt = re.sub(r'&[a-zA-Z0-9]+;', ' ', txt)
    txt = re.sub(r'https?://\S+|www\.\S+', ' ', txt)
    txt = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',' ',txt)
    txt = re.sub(r'\b\d{2,3}[-.\s]?\d{3,4}[-.\s]?\d{4}\b',' ',txt)
    txt = re.sub(r'\(\d{2}\)\s?\d{4,5}[-.\s]?\d{4}',' ',txt)
    txt = re.sub(r'\b\d{3}\.?\d{3}\.?\d{3}[-.]?\d{2}\b',' ',txt)
    txt = re.sub(r'\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}[-.]?\d{2}\b',' ',txt)
    txt = re.sub(r'[!?]{4,}','!!!',txt)
    txt = re.sub(r'\.{4,}','...',txt)
    txt = re.sub(r'-{3,}','--',txt)
    txt = re.sub(r'\s+',' ',txt)
    return txt.strip()
def clean_title_en(title: str) -> str:
    if not title: return ''
    title = normalize_text_en(title)
    title = re.sub(r'\s*[-–]\s*(promoção|oferta|novo|liquidação|desconto).*$', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\[(novo|promoção|oferta)\]','',title,flags=re.IGNORECASE)
    if title.isupper() and len(title)>3: title = title.lower().capitalize()
    if title and not title[0].isupper(): title = title[0].upper()+title[1:]
    return title.strip()
def clean_content_en(content: str) -> str:
    if not content: return ''
    content = normalize_text_en(content)
    content = DISCLAIMER_REGEX.sub(' ', content)
    sentences = re.split(r'[.!?]+', content)
    cleaned_sentences = []
    for s in sentences:
        st = s.strip()
        if st:
            if st.isupper() and len(st)>10: st = st.lower().capitalize()
            cleaned_sentences.append(st)
    content = '. '.join(cleaned_sentences)
    sentences = [s.strip() for s in content.split('.') if s.strip()]
    if sentences:
        first = sentences[0]; words = first.split()
        if len(words) > 30:
            first = ' '.join(words[:22])
            if not first.endswith(('.', '!', '?')): first += '.'
            sentences[0] = first
    content = '. '.join(sentences)
    paragraphs = content.split('\n'); final_paragraphs = []
    for para in paragraphs:
        p = para.strip()
        if len(p) <= 300:
            if p: final_paragraphs.append(p)
        else:
            parts = re.split(r'([.!?]+)', p)
            current = ''
            for i in range(0,len(parts),2):
                if i < len(parts):
                    sentence = parts[i]; punct = parts[i+1] if i+1 < len(parts) else ''
                    if len(current+sentence+punct) <= 300: current += sentence+punct
                    else:
                        if current: final_paragraphs.append(current.strip())
                        current = sentence+punct
            if current.strip(): final_paragraphs.append(current.strip())
    content = '\n'.join(final_paragraphs)
    content = re.sub(r'\n\s*\n','\n',content)
    content = re.sub(r'^\s*[-•*]\s*$','',content,flags=re.MULTILINE)
    return content.strip()
def count_tokens_en(text: str) -> int:
    if TOKENIZER:
        try: return len(TOKENIZER.encode(text))
        except: pass
    return len(text.split())
def create_compact_variant_en(content: str, max_words: int = 120) -> str:
    words = content.split()
    if len(words) <= max_words:
        return content
    cleaned_content = re.sub(r'\b(muito|super|extremamente|altamente|bastante)\s+','',content)
    cleaned_content = re.sub(r'\b(excelente|ótimo|perfeito|ideal|incrível)\s+(excelente|ótimo|perfeito|ideal|incrível)\b', r'\1', cleaned_content)
    sentences = [s.strip() for s in cleaned_content.split('.') if s.strip()]
    result_words = []
    for s in sentences:
        sw = s.split()
        if len(result_words)+len(sw) <= max_words:
            result_words.extend(sw)
        else:
            remaining = max_words - len(result_words)
            if remaining > 5:
                result_words.extend(sw[:remaining])
            break
    result = ' '.join(result_words)
    if not result.endswith(('.', '!', '?')):
        result += '.'
    return result
def build_examples_en(rows: List[Dict[str, Any]], cfg) -> List[Dict[str, Any]]:
    examples = []; stats = {'total_input':len(rows),'missing_fields':0,'short_title':0,'long_title':0,'short_content':0,'long_content':0,'low_letter_ratio':0,'duplicates':0,'valid':0}
    seen_hashes = set()
    for i,row in enumerate(rows):
        if not isinstance(row, dict) or 'title' not in row or 'content' not in row: stats['missing_fields'] += 1; continue
        title = str(row.get('title','')).strip(); content = str(row.get('content','')).strip()
        if not title or not content: stats['missing_fields'] += 1; continue
        clean_title_text = clean_title_en(title); clean_content_text = clean_content_en(content)
        if len(clean_title_text) < 2 or len(clean_title_text) > 120:
            stats['short_title' if len(clean_title_text)<2 else 'long_title'] += 1; continue
        content_tokens = count_tokens_en(clean_content_text)
        if content_tokens < cfg.min_tokens or content_tokens > cfg.max_tokens:
            stats['short_content' if content_tokens<cfg.min_tokens else 'long_content'] += 1; continue
        letters = sum(1 for c in clean_content_text if c.isalpha()); total_chars = len(clean_content_text)
        if total_chars>0 and letters/total_chars < 0.55: stats['low_letter_ratio'] += 1; continue
        content_hash = hashlib.blake2s((clean_title_text+clean_content_text).encode('utf-8')).hexdigest()
        if content_hash in seen_hashes: stats['duplicates'] += 1; continue
        seen_hashes.add(content_hash)
        instruction = INSTRUCTION_TEMPLATES_EN[i % len(INSTRUCTION_TEMPLATES_EN)]
        example_id = hashlib.blake2s((clean_title_text+clean_content_text+'original').encode('utf-8')).hexdigest()[:16]
        example = {'instruction':instruction,'input':clean_title_text,'output':clean_content_text,'id':example_id,'meta':{'source':'train_cleaned_unique.json','version':'v1','tokens':content_tokens,'variant':'original'}}
        examples.append(example); stats['valid'] += 1
        if cfg.augment:
            compact_content = create_compact_variant_en(clean_content_text)
            compact_tokens = count_tokens_en(compact_content)
            if compact_tokens >= cfg.min_tokens and compact_content != clean_content_text:
                compact_id = hashlib.blake2s((clean_title_text+compact_content+'compact').encode('utf-8')).hexdigest()[:16]
                examples.append({'instruction':instruction,'input':clean_title_text,'output':compact_content,'id':compact_id,'meta':{'source':'train_cleaned_unique.json','version':'v1','tokens':compact_tokens,'variant':'compact'}}); stats['valid'] += 1
    print('\nINFO: Filtering statistics:')
    print(f"  Total input: {stats['total_input']}")
    print(f"  Missing fields: {stats['missing_fields']}")
    print(f"  Short/long title: {stats['short_title']}/{stats['long_title']}")
    print(f"  Short/long content: {stats['short_content']}/{stats['long_content']}")
    print(f"  Low letter ratio: {stats['low_letter_ratio']}")
    print(f"  Duplicates: {stats['duplicates']}")
    print(f"  Valid examples: {stats['valid']}")
    return examples
def split_dataset_en(examples: List[Dict[str, Any]], seed: int) -> Tuple[List,List,List]:
    title_groups = {}
    for ex in examples:
        k = ex['input'].lower().strip(); title_groups.setdefault(k, []).append(ex)
    random.seed(seed); keys = list(title_groups.keys()); random.shuffle(keys)
    total = len(keys); train_size = int(total*0.9); valid_size = int(total*0.05)
    train_keys = keys[:train_size]; valid_keys = keys[train_size:train_size+valid_size]; test_keys = keys[train_size+valid_size:]
    train_examples=[]; [train_examples.extend(title_groups[k]) for k in train_keys]
    valid_examples=[]; [valid_examples.extend(title_groups[k]) for k in valid_keys]
    test_examples=[]; [test_examples.extend(title_groups[k]) for k in test_keys]
    random.shuffle(train_examples); random.shuffle(valid_examples); random.shuffle(test_examples)
    print('\nINFO: Dataset split:')
    print(f'  Train: {len(train_examples)} examples ({len(train_keys)} unique titles)')
    print(f'  Valid: {len(valid_examples)} examples ({len(valid_keys)} unique titles)')
    print(f'  Test: {len(test_examples)} examples ({len(test_keys)} unique titles)')
    return train_examples, valid_examples, test_examples
def write_jsonl_en(path: Path, rows: List[Dict[str, Any]]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+'\n')
    print(f'INFO: Wrote {len(rows)} examples to {path}')
def summarize_stats_en(examples: List[Dict[str, Any]]):
    if not examples: print('INFO: No examples to summarize'); return
    token_counts = [ex['meta']['tokens'] for ex in examples]
    title_counter = Counter(ex['input'] for ex in examples)
    print('\nINFO: Dataset statistics:')
    print(f'  Total examples: {len(examples)}')
    print('  Token distribution:')
    print(f'    p10: {statistics.quantiles(token_counts, n=10)[0]:.0f}')
    print(f'    p50: {statistics.median(token_counts):.0f}')
    print(f'    p90: {statistics.quantiles(token_counts, n=10)[8]:.0f}')
    print(f'    p95: {statistics.quantiles(token_counts, n=20)[18]:.0f}')
    print(f'    max: {max(token_counts)}')
    most_common = title_counter.most_common(10)
    duplicated = [(t,c) for t,c in most_common if c>1]
    if duplicated:
        print('  Top most frequent titles:')
        for t,c in duplicated: print(f"    '{t[:50]}...' ({c}x)")
    else:
        print('  All titles are unique')
def main_en():
    input_path = Path('D:/FIAP_TC3/train_cleaned_unique.json')
    outdir = Path('d:/FIAP_TC3/processed_dataset')
    parser = argparse.ArgumentParser(description='Prepare product dataset for LoRA fine-tuning')
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--min-tokens', type=int, default=15)
    parser.add_argument('--max-tokens', type=int, default=300)
    parser.add_argument('--augment', action='store_true')
    args = parser.parse_args()
    print('INFO: Starting processing with parameters:')
    print(f'  Input: {input_path}')
    print(f'  Output: {outdir}')
    print(f'  Min/Max tokens: {args.min_tokens}/{args.max_tokens}')
    print(f'  Augment: {args.augment}')
    print(f'  Seed: {args.seed}')
    raw = load_data_en(input_path)
    examples = build_examples_en(raw, args)
    if not examples: print('ERROR: No valid examples after filtering'); return
    summarize_stats_en(examples)
    train_ex, valid_ex, test_ex = split_dataset_en(examples, args.seed)
    write_jsonl_en(outdir/'train.jsonl', train_ex)
    write_jsonl_en(outdir/'valid.jsonl', valid_ex)
    write_jsonl_en(outdir/'test.jsonl', test_ex)
    print('\nINFO: Completed.')
main_en()

INFO: Starting processing with parameters:
  Input: D:\FIAP_TC3\train_cleaned_unique.json
  Output: d:\FIAP_TC3\processed_dataset
  Min/Max tokens: 15/300
  Augment: False
  Seed: 42
INFO: Loaded 1156410 records from file

INFO: Filtering statistics:
  Total input: 1156410
  Missing fields: 1
  Short/long title: 27/55771
  Short/long content: 70831/104393
  Low letter ratio: 5227
  Duplicates: 8
  Valid examples: 920152

INFO: Dataset statistics:
  Total examples: 920152
  Token distribution:
    p10: 27
    p50: 81
    p90: 222
    p95: 256
    max: 300
  Top most frequent titles:
    'Genuine Poulan Weedeater Part #...' (5x)
    'Frigidaire Door Bin for Refrigerator...' (4x)
    'Genuine Fuel Injection Air Flow Meter Boot...' (4x)
    'That's Not What I Meant!: How Conversational Style...' (2x)
    'Crafting and Executing Strategy: The Quest for Com...' (2x)
    'Tilly...' (2x)
    'Writers INC: A Student Handbook for Writing Learni...' (2x)
    'Until the Twelfth of Never: The Deadl

In [14]:
import pandas as pd, os
multi_path = 'd:/FIAP_TC3/processed_dataset/train.jsonl'
if os.path.exists(multi_path):
    df_multi = pd.read_json(multi_path, lines=True)
    display(df_multi.head(5))
else:
    print('Arquivo não encontrado:', multi_path)

,instruction,input,output,id,meta
0,Create a clear and attractive product descript...,Moving Blankets (4-pack) - Deluxe Mover - 72 X...,Heavy duty quilted furniture moving blanket is...,3651ccf4e064ce93,"{'source': 'train_cleaned_unique.json', 'versi..."
1,Create a clear and attractive product descript...,2 Bar Stool Black Elegant PU Leather Modern Ad...,Our kitchen / bar counter top bar stools will ...,70d44c5a319acf2e,"{'source': 'train_cleaned_unique.json', 'versi..."
2,Create a clear and attractive product descript...,Tuscan Sunset Arch with Oils and Cheese Decora...,"Handcrafted to the highest standards by ""Decor...",aaba5ef622f32628,"{'source': 'train_cleaned_unique.json', 'versi..."
3,Create a clear and attractive product descript...,Raybestos 550-1465 Professional Grade Suspensi...,Raybestos Professional Grade Sway Bar Bushings...,5ba4868554b67338,"{'source': 'train_cleaned_unique.json', 'versi..."
4,Create a clear and attractive product descript...,Ouran High School Host Club: School Logo Patch,Officially licensed Ouran High School Host Clu...,740eac6f57178c83,"{'source': 'train_cleaned_unique.json', 'versi..."
